# Setup

In [ ]:
cp MESA-Web_Job_04012666280/trimmed_history.data MESA-Web_Job_04012666280/history.data

## Imports

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import mesa_reader as mr
import numpy as np
import pandas as pd
import math

## history loading

In [ ]:
# Load your history file. Note directory location specified.
h = mr.MesaData('./MESA-Web_Job_04012666280/trimmed_history.data')
# https://www.pas.rochester.edu/~emamajek/EEM_dwarf_UBVIJHK_colors_Teff.txt
zams = pd.read_csv('mamajek_table.txt', sep='\s+')

# Load the entire directory
l = mr.MesaLogDir('./MESA-Web_Job_04012666280')

# Select a specific profile:
# Use l.profile_data() for the very last model,
# or l.profile_data(model_number=100) for a specific step.
p = l.profile_data(model_number=1000)

profiles_of_interest = {
    1: '1',
    2: '50',
    7: '275',
    9: '325',
    11: '362',
    14: '467',
    15: '476',
    17: '545',
    27: '1000',
}
print(f'history.profile columns: {h.bulk_names}\n\n ZAMS columns: {zams.columns}\n\n specfiic profile columns {p.bulk_names}')

## Filters

In [ ]:
# HR mask for 12M_sun
Amask = (h.log_Teff >= 4) & (h.log_Teff <=4.4) & (h.log_L <= 4.2)
Bmask = (h.log_Teff >= 3.7) & (h.log_Teff <= 4) & (h.log_L <= 4)
mask = Amask + Bmask

# ZAMS
zams = zams[zams['logL'] != '...']
zams = zams[zams['logT'] != '...']
zams['logL'] = zams['logL'].astype(float)
zams['logT'] = zams['logT'].astype(float)

## Constants

In [ ]:
G        = 6.674e-8
k_B      = 1.381e-16
m_H      = 1.673e-24
sigma_SB = 5.670e-5
M_sun    = 1.989e33
R_sun    = 6.957e10#cm
mu      = 0.62 # Fully ionized H/He approx
mu_e    = 1.15
a_rad   = 7.566e-15
c_light = 3e10
hbar    = 1.055e-27
m_e     = 9.109e-28

selected = {
    1:  ('Pre-MS Contraction', 'tab:purple'),
    7:  ('Main Sequence',      'tab:blue'),
    11: ('TAMS',               'tab:green'),
    27: ('He-burning / RSG',   'tab:red'),
}

burning_lines = {
    'H burning (pp/CNO)': (7.0,  'tab:blue',   '-'),
    'He burning (3α)':    (8.0,  'tab:orange',  '-'),
    'C burning':          (8.6,  'tab:green',   '-'),
}

virial_profiles = {7: ('Main Sequence', 'tab:blue'), 27: ('He-burning / RSG', 'tab:red')}

def get_p_age(prof_num):
    p   = l.profile_data(profile_number=prof_num)
    mn  = p.model_number
    age = h.star_age[h.model_number == mn][0]
    return p, age

# Bulk

## historyVars by star_age

In [ ]:
bulk_names = list(h.bulk_names)
bulk_names.remove('star_age')

COLS = 4
SECTION_SIZE = 60
sections = [bulk_names[i:i+SECTION_SIZE] for i in range(0, len(bulk_names), SECTION_SIZE)]

for sec_num, section in enumerate(sections):
    rows = math.ceil(len(section) / COLS)
    fig, axes = plt.subplots(rows, COLS, figsize=(COLS * 4, rows * 3))
    axes = axes.flatten()

    for i, name in enumerate(section):
        try:
            axes[i].plot(h.star_age, getattr(h, name), lw=1)
            axes[i].set_title(name, fontsize=8)
            axes[i].set_xlabel('Age (log yr)', fontsize=7)
            axes[i].tick_params(labelsize=6)
        except Exception as e:
            axes[i].set_title(f'{name}\n(error)', fontsize=7, color='red')

    for j in range(len(section), len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(f'History Variables: Section {sec_num + 1}/{len(sections)}', fontsize=10)
    plt.tight_layout()
#    plt.savefig(f'history_section_{sec_num + 1}.png', dpi=150, bbox_inches='tight')
    plt.show()

## historyVars by model_number

In [ ]:
bulk_names = list(h.bulk_names)
bulk_names.remove('model_number')

COLS = 10
SECTION_SIZE = 60
sections = [bulk_names[i:i+SECTION_SIZE] for i in range(0, len(bulk_names), SECTION_SIZE)]

for sec_num, section in enumerate(sections):
    rows = math.ceil(len(section) / COLS)
    fig, axes = plt.subplots(rows, COLS, figsize=(COLS * 4, rows * 3))
    axes = axes.flatten()

    for i, name in enumerate(section):
        try:
            axes[i].plot(h.model_number, getattr(h, name), lw=1)
            axes[i].set_title(name, fontsize=8)
            axes[i].set_xlabel('model_number', fontsize=7)
            axes[i].tick_params(labelsize=6)
            # Profiles of interests
            for prof_num in profiles_of_interest:
                p_temp = l.profile_data(profile_number=prof_num)
                axes[i].axvline(p_temp.model_number, color='red', alpha=0.5, lw=1)
                axes[i].text(p_temp.model_number, axes[i].get_ylim()[1], str(prof_num), fontsize=7, color='red')
        except Exception as e:
            axes[i].set_title(f'{name}\n(error)', fontsize=7, color='red')

    for j in range(len(section), len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(f'History Variables: Section {sec_num + 1}/{len(sections)}', fontsize=10)
    plt.tight_layout()
    plt.savefig(f'model_number_{sec_num + 1}.png', dpi=150, bbox_inches='tight')
    plt.show()

## profOfInterest profVars by mass

In [ ]:
PROP = 'mass'
COLS = 2
SECTION_SIZE = 60

for prof_num, label in profiles_of_interest.items():
    p = l.profile_data(profile_number=prof_num)
    model_num = p.model_number
    age = h.star_age[h.model_number == model_num][0]

    bulk_names = [n for n in p.bulk_names if n != PROP]
    rows = math.ceil(len(bulk_names) / COLS)
    fig, axes = plt.subplots(rows, COLS, figsize=(COLS * 4, rows * 3))
    axes = axes.flatten()

    for i, name in enumerate(bulk_names):
        try:
            axes[i].plot(getattr(p, PROP), getattr(p, name), lw=1)
            axes[i].set_title(name, fontsize=8)
            axes[i].set_xlabel(PROP, fontsize=7)
            axes[i].tick_params(labelsize=6)
        except Exception as e:
            axes[i].set_title(f'{name}\n(error)', fontsize=7, color='red')

    for j in range(len(section), len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(
        f'Profile {prof_num} | Model {model_num} | Age: {age:.3e} yr', fontsize=11
    )
    plt.tight_layout(rect=[0, 0, 1, 0.97])
#   fig.savefig(f'profile_{prof_num:02d}_model_{model_num}.png', dpi=150, bbox_inches='tight')
    plt.show()

# Specific Profile/History plots

## (Profile) Internal Structure: Log T by log Rho

In [ ]:
plt.figure(figsize=(8, 6))

# Plot logT vs logRho
plt.plot(p.logRho, p.logT, color='black', label='Stellar Interior')

plt.xlabel(r'$\log_{10}(\rho  [\rm{g\,cm}^{-3}])$')
plt.ylabel(r'$\log_{10}(T  [\rm{K}])$')
plt.title(f'Internal Structure (Model {p.model_number})')
plt.grid(True, linestyle=':', alpha=0.7)

plt.legend()
plt.show()

## (History) Log Age by (Mass and Log_center_p) Luminosity

In [ ]:
fig, ax = plt.subplots()
ax.plot(h.star_age, h.log_center_P)
ax.set_xlabel('Age (years)')
ax.set_ylabel('Log(Center P)')
ax.set_title('Log Centeral Pressure by Age')
plt.savefig('age_by_centerP.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.plot(h.star_age, h.star_mass)
ax.set_xlabel('Age (years)')
ax.set_ylabel('mass')
ax.set_title('Mass by Age')
plt.savefig('age_by_mass.png', dpi=150, bbox_inches='tight')
plt.show()

## (History) Model Number by tri_alpa

In [ ]:
fig, ax = plt.subplots()
ax.plot(h.model_number, h.tri_alfa)
ax.set_xlabel('model number')
ax.set_ylabel('tri_alpa')
ax.set_title('Triple-alpha burning rate over time')

# Profiles of interests
for prof_num in profiles_of_interest:
    p_temp = l.profile_data(profile_number=prof_num)
    ax.axvline(p_temp.model_number, color='red', alpha=0.5, lw=1)
    ax.text(p_temp.model_number, ax.get_ylim()[1], str(prof_num), fontsize=7, color='red')

plt.show()

## (History) Model Number by pp

In [ ]:
fig, ax = plt.subplots()
ax.plot(h.model_number, h.pp)
ax.set_xlabel('model number')
ax.set_ylabel('pp')
ax.set_title('pp burning rate over time')

# Profiles of interests
for prof_num in profiles_of_interest:
    p_temp = l.profile_data(profile_number=prof_num)
    ax.axvline(p_temp.model_number, color='red', alpha=0.5, lw=1)
    ax.text(p_temp.model_number, ax.get_ylim()[1], str(prof_num), fontsize=7, color='red')

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

## (History) log T_eff by Log L

In [ ]:
fig, ax = plt.subplots()
ax.plot(h.log_Teff, h.log_L)
ax.set_xlabel('Log T_eff')
ax.set_ylabel('Log L')
ax.set_title('Hayashi zone')

# Profiles of interests
for prof_num in profiles_of_interest:
    p_temp = l.profile_data(profile_number=prof_num)
    ax.axvline(p_temp.model_number, color='red', alpha=0.5, lw=1)
    ax.text(p_temp.model_number, ax.get_ylim()[1], str(prof_num), fontsize=7, color='red')

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

## (History) Nuclear burning and Central temperature history

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)
# Mass fractions
axes[0].plot(h.model_number, h.center_h1, label='center_h1', lw=1.5)
axes[0].plot(h.model_number, h.center_he4, label='center_he4', lw=1.5)
axes[0].set_ylabel('Mass Fraction')
axes[0].set_title('Core Composition')
axes[0].legend(fontsize=8)
axes[0].grid(True, linestyle='--', alpha=0.5)

# Nuclear Burning
axes[1].plot(h.model_number, h.cno, label='cno', lw=1.5)
axes[1].plot(h.model_number, h.tri_alfa, label='tri_alfa', lw=1.5)
axes[1].set_ylabel('log Burning Rate')
axes[1].set_title('Nuclear Burning Rates')
axes[1].legend(fontsize=8)
axes[1].grid(True, linestyle='--', alpha=0.5)

# Central Temperature
axes[2].plot(h.model_number, h.log_center_T, color='black', lw=1.5)
axes[2].set_ylabel('log_center_T')
axes[2].set_title('Central Temperature Evolution')
axes[2].grid(True, linestyle='--', alpha=0.5)

# Interesting Model Numbers
for prof_num, model_str in profiles_of_interest.items():
    for ax in axes:
        ax.axvline(int(model_str), color='red', alpha=0.5, lw=1)
        ax.text(int(model_str), ax.get_ylim()[1], str(prof_num), fontsize=7, color='red', ha='center')

axes[2].set_xlabel('Model Number')
fig.suptitle('Evolution', fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('Evolution.png', dpi=150, bbox_inches='tight')
plt.show()

# Hertzsprung-Russell Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(zams['logT'], zams['logL'], 'b--', label='ZAMS (Pecaut & Mamajek 2013)', zorder=1) # ZAMS reference line

profiles_of_interest = {
    1: '1',
    7: '275',
    11: '362',
    27: '1000',
}

short_labels = {
    1: 'Start/Contraction',
    7: 'Main Sequence',
    11: 'TAMS',
    27: 'Adv-burning',
}

offsets = {
    1:  (6, -15),
    7:  (-50, -15),
    11: (-65, 15),
    27: (-30, 25),
}

dt = np.diff(h.star_age)
dt = np.append(dt, dt[-1])
log_dt = np.log10(dt + 1)

sc = ax.scatter(h.log_Teff, h.log_L, c=log_dt, cmap='plasma', s=5, zorder=3)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('log$_{10}$(years per model step)', fontsize=10)

for prof_num, model_str in profiles_of_interest.items():
    model_num = int(model_str)
    idx = np.where(h.model_number == model_num)[0]
    if len(idx) == 0:
        continue
    idx = idx[0]
    x = h.log_Teff[idx]
    y = h.log_L[idx]
    ax.scatter(x, y, color='red', s=10, zorder=5)
    ax.annotate(
        short_labels[prof_num],
        xy=(x, y),
        xytext=offsets[prof_num],
        textcoords='offset points',
        fontsize=7,
        arrowprops=dict(arrowstyle='-', color='gray', lw=0.5)
    )

ax.invert_xaxis()  # Hotter stars on the left
ax.set_xlabel(r'$\log_{10}(T_{\rm eff} [ {\rm K}])$')
ax.set_ylabel(r'$\log_{10}(L / L_{\odot})$')
ax.set_title('Hertzsprung-Russell Diagram')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.6)

plt.savefig('hr_diagram.png', dpi=150, bbox_inches='tight')
plt.show()

# Profiles of Interest

## Q1 & Q2 Power Sources: nuclear burning rates + fuel abundances

In [ ]:
# Q1 Q2, Power Sources: nuclear burning rates + fuel abundances
fig, axes = plt.subplots(4, 3, figsize=(16, 18))

for row, (prof_num, (label, color)) in enumerate(selected.items()):
    p, age = get_p_age(prof_num)
    ax = axes[row, 0]

    # Energy Generation Rates
    ax.plot(p.mass, p.pp,       lw=1.5, label='pp chain',   color='tab:blue')
    ax.plot(p.mass, p.cno,      lw=1.5, label='CNO',        color='tab:orange')
    ax.plot(p.mass, p.tri_alfa, lw=1.5, label='triple-α',   color='tab:red')
    ax.plot(p.mass, np.abs(p.eps_grav), lw=1.5, ls='--',
            label=r'|ε$_{grav}$|', color='gray')
    ax.set_yscale('symlog', linthresh=1e-3)
    ax.set_xlabel(r'Enclosed Mass ($M_\odot$)')
    ax.set_ylabel(r'ε (erg g$^{-1}$ s$^{-1}$)')
    ax.set_title(f'P{prof_num}: {label} | {age:.2e} yr\nBurning Rates', fontsize=9)
    ax.legend(fontsize=7)
    ax.grid(True, linestyle=':', alpha=0.5)

    # Fuel Abundances
    ax = axes[row, 1]
    ax.plot(p.mass, p.h1,  lw=1.5, label=r'$^1$H',   color='tab:blue')
    ax.plot(p.mass, p.he4, lw=1.5, label=r'$^4$He',  color='tab:orange')
    ax.plot(p.mass, p.c12, lw=1.5, label=r'$^{12}$C',color='tab:green')
    ax.plot(p.mass, p.o16, lw=1.5, label=r'$^{16}$O',color='tab:red')
    ax.set_xlabel(r'Enclosed Mass ($M_\odot$)')
    ax.set_ylabel('Mass Fraction')
    ax.set_title('Composition', fontsize=9)
    ax.legend(fontsize=7)
    ax.grid(True, linestyle=':', alpha=0.5)

    # Temp + Density profile
    ax = axes[row, 2]
    ax2 = ax.twinx()
    ax.plot(p.mass,  p.logT,   lw=1.5, color='tab:red',   label='log T')
    ax2.plot(p.mass, p.logRho, lw=1.5, color='tab:blue', ls='--', label='log ρ')
    ax.set_xlabel(r'Enclosed Mass ($M_\odot$)')
    ax.set_ylabel('log T (K)', color='tab:red')
    ax2.set_ylabel(r'log ρ (g cm$^{-3}$)', color='tab:blue')
    ax.set_title('T & ρ structure', fontsize=9)
    ax.axhline(7.0, color='tab:orange', ls=':', lw=1, alpha=0.7)
    ax.axhline(8.0, color='tab:red',    ls=':', lw=1, alpha=0.7)
    ax.text(p.mass.max()*0.55, 7.05, 'T$_{CNO}$', fontsize=7, color='tab:orange')
    ax.text(p.mass.max()*0.55, 8.05, r'T$_{3\alpha}$', fontsize=7, color='tab:red')
    lines1, labs1 = ax.get_legend_handles_labels()
    lines2, labs2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, labs1+labs2, fontsize=7)
    ax.grid(True, linestyle=':', alpha=0.5)

fig.suptitle('Q1 & Q2: Power Sources and Conditions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('q1_q2_power_sources.png', dpi=150, bbox_inches='tight')
plt.show()

## Q3 What is pressure mainly due to?

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (prof_num, (label, color)) in zip(axes, selected.items()):
    p, age = get_p_age(prof_num)

    beta     = p.pgas / (p.pgas + p.prad)
    beta_rad = p.prad / (p.pgas + p.prad)

    ax.stackplot(p.mass,
                 beta, beta_rad,
                 labels=[r'Gas ($\beta$)', r'Radiation ($1-\beta$)'],
                 colors=['tab:blue', 'tab:orange'], alpha=0.6)

    ax2 = ax.twinx()
    ax2.plot(p.mass, p.eta, color='tab:red', lw=1.2, ls='--', label='η (degeneracy)')
    ax2.axhline(0, color='tab:red', lw=0.8, ls=':')
    ax2.set_ylabel('η (degeneracy param)', color='tab:red', fontsize=8)
    ax2.tick_params(axis='y', labelcolor='tab:red', labelsize=7)

    ax.set_title(f'P{prof_num}: {label}\nAge = {age:.2e} yr', fontsize=9)
    ax.set_xlabel(r'Enclosed Mass ($M_\odot$)')
    ax.set_ylabel('Fraction of Total Pressure')
    ax.set_ylim(0, 1.05)
    lines1, labs1 = ax.get_legend_handles_labels()
    lines2, labs2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, labs1+labs2, fontsize=7, loc='lower right')
    ax.grid(True, linestyle=':', alpha=0.4)

fig.suptitle('Q3: Pressure Sources: Gas vs Radiation vs Degeneracy', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('q3_pressure.png', dpi=150, bbox_inches='tight')
plt.show()

## Q4 Convective via Log_D_conv

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 18))

for row, (prof_num, (label, color)) in enumerate(selected.items()):
    p, age = get_p_age(prof_num)

    ax = axes[row, 0]
    p.gradr = np.clip(p.gradr, -999, 5)
    ax.plot(p.mass, p.gradr, color=color,   lw=1.5, label=r'$\nabla_\mathrm{rad}$')
    ax.plot(p.mass, p.grada, color='black', lw=1.5, ls='--', label=r'$\nabla_\mathrm{ad}$')
    ax.fill_between(p.mass, p.gradr, p.grada,
                    where=(p.gradr > p.grada),
                    alpha=0.3, color=color, label='Convective zone')
    ax.set_xlabel(r'Enclosed Mass ($M_\odot$)')
    ax.set_ylabel(r'$\nabla$')
    ax.set_ylim(-0.02, max(0.6, float(np.percentile(p.gradr, 99))*1.1))
    ax.set_title(f'P{prof_num}: {label} | {age:.2e} yr\nSchwarzschild Criterion', fontsize=9)
    ax.legend(fontsize=7)
    ax.grid(True, linestyle=':', alpha=0.5)

    ax = axes[row, 1]
    D_conv = np.where(p.log_D_conv < -10, np.nan, np.array(p.log_D_conv, dtype=float))

    ax.plot(p.mass, D_conv, color=color, lw=1.5)
    ax.fill_between(p.mass, D_conv, 0,
                    where=(D_conv > 1),
                    alpha=0.3, color=color, label='Convective (D > 10 cm² s⁻¹)')
    ax.axhline(0, color='gray', ls=':', lw=1)
    ax.set_xlabel(r'Enclosed Mass ($M_\odot$)')
    ax.set_ylabel(r'$\log_{10}\, D_\mathrm{conv}$ (cm² s⁻¹)')
    ax.set_title(f'P{prof_num}: Convective Diffusivity', fontsize=9)
    ax.legend(fontsize=7)
    ax.grid(True, linestyle=':', alpha=0.5)

    fig.suptitle('Q4: Convective Zones', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('q4_convection.png', dpi=150, bbox_inches='tight')
plt.show()

## Q4 Convection via Kipper

In [ ]:
import sys
sys.path.insert(0, './mkipp')
import mkipp
import kipp_data
import mesa_data as mkipp_mesa
from matplotlib.patches import PathPatch

LOGS_DIR  = './MESA-Web_Job_04012666280'
HIST_PATH = f'{LOGS_DIR}/history.data'

def dconv_extractor(identifier, log10_on_data, prof, return_data_columns=False):
    if return_data_columns:
        return ['log_D_conv']
    data = prof.get('log_D_conv')
    data = np.where(data < -10, 0, data)
    return data

fig, ax = plt.subplots(figsize=(12, 6))
kp = mkipp.kipp_plot(mkipp.Kipp_Args(
    logs_dirs     = [LOGS_DIR],
    history_paths = [HIST_PATH],
    identifier    = 'log_D_conv',
    extractor     = dconv_extractor,
    log10_on_data = False,
    show_conv     = False,
    show_semi     = False,
    show_over     = False,
    core_masses   = ['He', 'C', 'O'],
    decorate_plot = False,
    save_file     = False,
), axis=ax)

cbar = fig.colorbar(kp.contour_plot, ax=ax, pad=0.02)
cbar.set_label(r'$\log_{10} D_\mathrm{conv}$ (cm² s⁻¹)', fontsize=10)

ax.set_title(r'Q4: Convective Structure over Evolution (12 $M_\odot$)', fontsize=12, fontweight='bold')

profile_markers = {
    1:  (1,    'Pre-MS',  'tab:purple'),
    7:  (275,  'MS',      'tab:blue'),
    11: (362,  'TAMS',    'tab:green'),
    27: (1000, 'RSG',     'tab:red'),
}

ax.set_xlabel('Model Number', fontsize=11)
ax.set_ylabel(r'Enclosed Mass ($M_\odot$)', fontsize=11)

fig.canvas.draw()
ymin, ymax = ax.get_ylim()
xmin, xmax = ax.get_xlim()
for prof_num, (mn, plabel, pcolor) in profile_markers.items():
    ax.axvline(mn, color=pcolor, lw=1.5, ls='-', alpha=0.9, zorder=5)
    ax.text(mn, ymax * 0.92, plabel,
            fontsize=8, color=pcolor, ha='center', va='top', zorder=6,
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1))

legend_handles = [
    mpatches.Patch(color='black',      label='Stellar surface (mass loss)'),
    mpatches.Patch(color='blue',       label='He core boundary'),
    mpatches.Patch(color='red',        label='C core boundary'),
    mpatches.Patch(color='green',      label='O core boundary'),
    mpatches.Patch(color='tab:purple', label='P1:  Pre-MS'),
    mpatches.Patch(color='tab:blue',   label='P7:  Main Sequence'),
    mpatches.Patch(color='tab:green',  label='P11: TAMS'),
    mpatches.Patch(color='tab:red',    label='P27: Red Supergiant'),
]
ax.legend(handles=legend_handles, fontsize=8, loc='upper left', framealpha=0.8)

plt.tight_layout()
plt.savefig('q4_kipp_dconv.png', dpi=150, bbox_inches='tight')
plt.show()

## Q5 Mass Loss

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(h.model_number, h.star_mass, color='black', lw=1.5)
axes[0].set_ylabel(r'$M_\star$ ($M_\odot$)')
axes[0].set_title('Stellar Mass over Time')
axes[0].grid(True, linestyle=':', alpha=0.5)

axes[1].plot(h.model_number, np.log10(np.abs(h.star_mdot) + 1e-20),
             color='tab:red', lw=1.5)
axes[1].set_ylabel(r'$\log_{10}|\dot{M}|$ ($M_\odot$ yr$^{-1}$)')
axes[1].set_title(r'Mass-Loss Rate $\dot{M}$')
axes[1].set_xlabel('Model Number')
axes[1].grid(True, linestyle=':', alpha=0.5)

for prof_num, (label, color) in selected.items():
    p_temp = l.profile_data(profile_number=prof_num)
    for ax in axes:
        ax.axvline(p_temp.model_number, color=color, alpha=0.7, lw=1.2, ls='--')
        ax.text(p_temp.model_number, ax.get_ylim()[1]*1.0,
                f'P{prof_num}', fontsize=7, color=color, ha='center', va='top')

total_lost = h.star_mass[0] - h.star_mass[-1]
axes[0].text(0.02, 0.05,
             f'Total mass lost: {total_lost:.3f} $M_\odot$\n'
             r'(Driven by radiation pressure / stellar winds)',
             transform=axes[0].transAxes, fontsize=9,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

fig.suptitle('Q5: Mass Loss History', fontsize=13, fontweight='bold')
plt.savefig('q5_mass_loss.png', dpi=150, bbox_inches='tight')
plt.show()

## Q6 Log T vs log Rho

In [ ]:
rho_grid = np.linspace(-9, 8, 500)
rho_lin  = 10**rho_grid

T_gas_rad = (3 * k_B * rho_lin / (a_rad * mu * m_H))**(1/3)
logT_gas_rad = np.log10(T_gas_rad)

logT_degen = np.log10(2.3e5) + (2/3) * (rho_grid - np.log10(mu_e))

log_rho_rel_deg = np.log10(2e6 * mu_e)

fig, ax = plt.subplots(figsize=(12, 9))

ax.plot(rho_grid, logT_gas_rad, 'k-',  lw=1.5, label=r'$P_\mathrm{gas} = P_\mathrm{rad}$')
ax.plot(rho_grid, logT_degen,   'k--', lw=1.5, label=r'$P_\mathrm{gas} = P_\mathrm{deg,NR}$')
ax.axvline(log_rho_rel_deg, color='k', ls=':', lw=1.5, label=r'Rel. degeneracy ($\rho/\mu_e = 2\times10^6$)')

for blabel, (logT_burn, bcolor, bls) in burning_lines.items():
    ax.axhline(logT_burn, color=bcolor, ls=bls, lw=1.2, alpha=0.7, label=blabel)

for prof_num, (label, color) in selected.items():
    p, age = get_p_age(prof_num)
    ax.plot(p.logRho, p.logT, lw=2, color=color,
            label=f'P{prof_num}: {label} ({age:.1e} yr)')
    ax.scatter(p.logRho[-1], p.logT[-1], color=color, s=60, zorder=5)
    ax.scatter(p.logRho[0],  p.logT[0],  color=color, s=30, marker='x', zorder=5)

ax.set_xlim(-9, 9)
ax.set_ylim(3.5, 10)
ax.set_xlabel(r'$\log_{10}(\rho\ [\mathrm{g\ cm}^{-3}])$', fontsize=12)
ax.set_ylabel(r'$\log_{10}(T\ [\mathrm{K}])$', fontsize=12)
ax.set_title('Q6: Internal Structure in log T - log rho plane\n'
             'filled circle = center, x = surface', fontsize=12)
ax.legend(fontsize=8, loc='upper left')
ax.grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig('q6_logT_logRho.png', dpi=150, bbox_inches='tight')
plt.show()

## Q7 Virial Theorm

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (prof_num, (label, color)) in zip(axes, virial_profiles.items()):
    p, age = get_p_age(prof_num)

    m_g = p.mass * M_sun
    r_cm = p.radius * R_sun

    m_g_rev   = m_g[::-1]
    r_rev     = r_cm[::-1]
    dm        = np.diff(m_g_rev)

    m_mid     = 0.5 * (m_g_rev[:-1]  + m_g_rev[1:])
    r_mid     = 0.5 * (r_rev[:-1]    + r_rev[1:])
    dE_grav   = -G * m_mid / r_mid * dm
    E_grav    = np.sum(dE_grav)

    energy_rev = p.energy[::-1]
    u_mid      = 0.5 * (energy_rev[:-1] + energy_rev[1:])
    dE_therm   = u_mid * dm
    E_therm    = np.sum(dE_therm)

    virial_ratio = -2 * E_therm / E_grav

    E_grav_cum  = np.cumsum(dE_grav)
    E_therm_cum = np.cumsum(dE_therm)
    mass_mid_Msun = m_mid / M_sun

    ax.plot(mass_mid_Msun, E_grav_cum,       color='tab:blue',   lw=1.5, label=r'$E_\mathrm{grav}$ (cumulative)')
    ax.plot(mass_mid_Msun, E_therm_cum,      color='tab:red',    lw=1.5, label=r'$E_\mathrm{therm}$ (cumulative)')
    ax.plot(mass_mid_Msun, -2*E_therm_cum,   color='tab:orange', lw=1.5, ls='--', label=r'$-2E_\mathrm{therm}$ (should equal $E_\mathrm{grav}$)')

    ax.set_xlabel(r'Enclosed Mass ($M_\odot$)')
    ax.set_ylabel('Energy (erg)')
    ax.set_title(f'P{prof_num}: {label}\n'
                 f'2E_therm/|E_grav| = {virial_ratio:.3f}', fontsize=9)
    ax.legend(fontsize=7)
    ax.grid(True, linestyle=':', alpha=0.5)
    ax.axhline(0, color='black', lw=0.8, ls=':')

    ax.text(0.05, 0.07,
            f'Virial ratio = {virial_ratio:.3f}\n'
            f'E_grav = {E_grav:.3e} erg\n'
            f'E_therm = {E_therm:.3e} erg',
            transform=ax.transAxes, fontsize=8,
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

fig.suptitle(r'Q7: Virial Theorem: $2E_{therm} + E_{grav} = 0$?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('q7_virial.png', dpi=150, bbox_inches='tight')
plt.show()